<a href="https://colab.research.google.com/github/Mogi21/nlp-engineering-portfolio/blob/main/03_ner_bias_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade setuptools pip
!pip install evaluate datasets transformers

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import torch
import datasets
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)

# Assuming a local helper module containing dataset loaders and metric functions
from helpers import (
    tokenize_and_align_labels,
    label_list,
    label_encoding_dict,
    compute_metrics,
    load_gendered_samples,
    load_ethnic_datasets
)

# ==========================================
# 1. CONFIGURATION & MODEL INGESTION
# ==========================================
os.environ['WANDB_DISABLED'] = 'true'
MODEL_TYPE = "bert-base-uncased"
CUSTOM_MODEL_PATH = "./sketchy_bert_ner.model"
BATCH_SIZE = 16

print("Loading Tokenizer and Custom NER Model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_TYPE)
model = AutoModelForTokenClassification.from_pretrained(CUSTOM_MODEL_PATH, num_labels=len(label_list))

# Align label encodings
model.config.label2id = label_encoding_dict
model.config.id2label = {idx: label for idx, label in enumerate(label_list)}

data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ==========================================
# 2. BASELINE & OUT-OF-DOMAIN GENERALIZATION
# ==========================================
print("\n--- Evaluating on In-Domain Validation Data ---")
sketchy_val = datasets.Dataset.load_from_disk('sketchy_ner_data.val')
sketchy_val_tokenized = sketchy_val.map(tokenize_and_align_labels(tokenizer), batched=True)

in_domain_metrics = trainer.predict(sketchy_val_tokenized).metrics
print(f"In-Domain F1 Score: {in_domain_metrics.get('test_f1', 'N/A')}")

print("\n--- Evaluating Out-of-Domain Generalization (CoNLL 2003) ---")
conll = datasets.load_dataset('conll2003')
conll_val_tokenized = conll['validation'].map(tokenize_and_align_labels(tokenizer), batched=True)

prediction_output = trainer.predict(conll_val_tokenized)
print(f"Out-of-Domain F1 Score: {prediction_output.metrics.get('test_f1', 'N/A')}")

# ==========================================
# 3. ERROR ANALYSIS: CONFUSION MATRIX
# ==========================================
print("\n--- Generating Confusion Matrix for Error Analysis ---")
logits, labels = prediction_output.predictions, prediction_output.label_ids

# Convert logits to absolute predictions
predictions = np.argmax(logits, axis=-1)

# Filter out ignored index (usually -100 for special tokens in PyTorch)
mask = labels != -100
y_trues = labels[mask]
y_preds = predictions[mask]

# Map integer IDs back to string labels for the matrix
y_trues_labels = [model.config.id2label[t] for t in y_trues]
y_preds_labels = [model.config.id2label[p] for p in y_preds]

cm = confusion_matrix(y_trues_labels, y_preds_labels, labels=label_list)

plt.figure(figsize=(10, 8))
ax = sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', vmax=3000, vmin=0)
ax.set_xlabel('Predicted Labels')
ax.set_ylabel('True Labels')
ax.xaxis.set_ticklabels(label_list, rotation=45)
ax.yaxis.set_ticklabels(label_list, rotation=0)
plt.title("NER Confusion Matrix: CoNLL 2003 Evaluation")
plt.tight_layout()
plt.show()

# ==========================================
# 4. DEMOGRAPHIC AUDIT: GENDER BIAS
# ==========================================
print("\n--- Conducting Gender Bias Audit ---")
# Dataset from Mehrabi et al. (2020)
female_samples, male_samples = load_gendered_samples()

# Initialize a standard inference pipeline
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

def evaluate_gender_recall(samples, target_entity="PER"):
    """Calculates what percentage of gendered samples successfully triggered a Person (PER) tag."""
    successful_detections = 0
    predictions = ner_pipeline(samples, batch_size=BATCH_SIZE)

    for pred_list in predictions:
        # Check if the target entity (e.g., PER) was detected anywhere in the sample
        if any(entity['entity_group'] == target_entity for entity in pred_list):
            successful_detections += 1

    return successful_detections / len(samples)

female_recall = evaluate_gender_recall(female_samples)
male_recall = evaluate_gender_recall(male_samples)

print(f"Female Name Detection Recall: {female_recall:.2%}")
print(f"Male Name Detection Recall:   {male_recall:.2%}")
print(f"Disparate Impact Ratio:       {female_recall / male_recall if male_recall > 0 else 0:.2f}")

# ==========================================
# 5. DEMOGRAPHIC AUDIT: ETHNIC BIAS
# ==========================================
print("\n--- Conducting Ethnic Bias Audit ---")
# Dataset processed based on Mishra et al. (2020)
# Categories: b (Black), h (Hispanic), w (White), m (Muslim) crossed with m/w (Man/Woman)
ethnic_datasets = load_ethnic_datasets()

results = []
for category, dataset_split in ethnic_datasets.items():
    tokenized_split = dataset_split.map(tokenize_and_align_labels(tokenizer), batched=True)
    metrics = trainer.predict(tokenized_split).metrics

    results.append({
        "Demographic Category": category,
        "Precision": metrics.get('test_precision', 0),
        "Recall": metrics.get('test_recall', 0),
        "F1 Score": metrics.get('test_f1', 0)
    })

results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)
print("\nModel Performance Across Ethnic/Gender Subsets:")
print(results_df.to_string(index=False))

ModuleNotFoundError: No module named 'helpers'